In [ ]:
import os, sys, time, json, glob, subprocess, shutil, signal, threading, urllib.request
T0 = time.time()
OUT = '/kaggle/working/e042'; os.makedirs(OUT, exist_ok=True)
# Hard cap on execution time (the watchdog in the server cell kills the server here; the finish cell still runs).
# Kaggle may hold a push in "RUNNING" for hours before this cell starts: T0 is the real start.
HARD_STOP_S = 11400                  # run 9: ≈ 3.2 h, two tool points (run 8: 5400, run 7: 5600, run 6: 6000, run 5: 7200)
NEW_PHASE_UNTIL_S = HARD_STOP_S      # run 4 cells: no new phase after this

def log(*a):
    s = f"[{time.time() - T0:7.0f}s] " + " ".join(str(x) for x in a)
    print(s, flush=True)
    with open(f'{OUT}/run.log', 'a') as fh:
        fh.write(s + '\n')

def sh(cmd, timeout=None, env=None):
    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout, env=env)
    except subprocess.TimeoutExpired:
        return 124, f'TIMEOUT after {timeout}s'
    return r.returncode, (r.stdout or '') + (r.stderr or '')

def time_left():
    return NEW_PHASE_UNTIL_S - (time.time() - T0)

# Host driver libcuda first (the image also carries an older compat libcuda); nvidia-smi needs it too.
real = '/usr/local/nvidia/lib64'
if os.path.exists(real + '/libcuda.so'):
    os.environ['LD_LIBRARY_PATH'] = real + ':' + os.environ.get('LD_LIBRARY_PATH', '')
    os.environ['LD_PRELOAD'] = real + '/libcuda.so'
for cmd in ['nvidia-smi', 'nvidia-smi topo -m', 'nvidia-smi -q -d CLOCK,POWER,TEMPERATURE,PERFORMANCE | head -150',
            'free -g', 'df -h /tmp /kaggle/working', 'nproc', "grep -m1 'model name' /proc/cpuinfo",
            'cat /sys/fs/cgroup/memory.max 2>/dev/null', 'uv --version', 'python --version']:
    rc, out = sh(cmd, timeout=120)
    open(f'{OUT}/env.txt', 'a').write(f'$ {cmd}\n{out}\n')
    log('$', cmd, '\n' + out[-3000:])

In [ ]:
def find_one(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    return hits[0] if hits else None
WHEELS = os.path.dirname(find_one('/kaggle/input/**/vllm-0.28.1rc1*.whl'))
KIT = os.path.dirname(find_one('/kaggle/input/**/model-e64-meta.tgz'))
log('wheels', WHEELS, 'kit', KIT)
V = '/tmp/dsv41/venv'; PY = f'{V}/bin/python'; SP = f'{V}/lib/python3.12/site-packages'
os.makedirs('/tmp/dsv41', exist_ok=True)
t = time.time()
if shutil.which('uv'):
    # --no-deps: the lock is the box's complete freeze (the fork's vllm names flashinfer by URL; uv rejects that).
    rc, out = sh(f'uv venv --python {sys.executable} {V} && uv pip install --python {PY} --no-index --no-deps '
                 f'--find-links {WHEELS} -r {WHEELS}/requirements.lock', timeout=3600)
else:
    rc, out = sh(f'{sys.executable} -m venv {V} && {PY} -m pip install --no-index --no-deps --find-links {WHEELS} '
                 f'-r {WHEELS}/requirements.lock', timeout=3600)
log('offline install rc', rc, f'{time.time() - t:.0f}s', out[-1500:])
assert rc == 0
for cmd in [f'{PY} -I {KIT}/apply_patches.py {SP}/vllm',
            f'mkdir -p ~/.cache/flashinfer && tar xzf {KIT}/flashinfer_jit_cache_fork_sm89.tgz -C ~/.cache/flashinfer',
            f'mkdir -p /tmp/dsv41/model-e64 && tar xzf {KIT}/model-e64-meta.tgz -C /tmp/dsv41/model-e64 && ls -la /tmp/dsv41/model-e64',
            f'cp -r /tmp/dsv41/model-e64 /tmp/dsv41/model-e48 && cp {KIT}/e48_config.json /tmp/dsv41/model-e48/config.json && '
            f'cp {KIT}/e48_synthetic_manifest.json /tmp/dsv41/model-e48/synthetic_manifest.json && '
            f'grep -o \'"n_routed_experts": [0-9]*\' /tmp/dsv41/model-e48/config.json']:
    rc, out = sh(cmd, timeout=600)
    log('$', cmd, 'rc', rc, out[-1200:])
    assert rc == 0

# PR #154 (env-gated): the file pre-patched from the box's original, installed only if the wheel's file is that
# original (no dependency on `patch` in the image).
import hashlib
IDX = f'{SP}/vllm/model_executor/kernels/attention/dsa/dsv41_indexer.py'
orig_ok = hashlib.sha256(open(IDX, 'rb').read()).hexdigest() == open(f'{KIT}/dsv41_indexer.orig.sha256').read().split()[0]
if orig_ok:
    shutil.copy(f'{KIT}/dsv41_indexer.pr154.py', IDX)
log('PR #154 installed (env-gated):', orig_ok)

# CUDA 13 toolkit from the pip wheels behind a CUDA_HOME shim (no nvcc in the image), for any FlashInfer JIT not
# covered by the restored cache (kaggle/src/solver.cuda_toolkit_env, pointed at the venv).
cu = f'{SP}/nvidia/cu13'; shim = '/tmp/cuda13'
os.makedirs(f'{shim}/lib64', exist_ok=True)
for d in ('bin', 'include', 'nvvm', 'lib', 'share'):
    if os.path.exists(f'{cu}/{d}') and not os.path.lexists(f'{shim}/{d}'):
        os.symlink(f'{cu}/{d}', f'{shim}/{d}')
for f in os.listdir(f'{cu}/lib'):
    if not os.path.lexists(f'{shim}/lib64/{f}'):
        os.symlink(f'{cu}/lib/{f}', f'{shim}/lib64/{f}')
for name in ('cudart', 'nvrtc'):
    real = sorted(glob.glob(f'{cu}/lib/lib{name}.so.*'))
    if real and not os.path.lexists(f'{shim}/lib64/lib{name}.so'):
        os.symlink(real[0], f'{shim}/lib64/lib{name}.so')
# FlashInfer JIT links with -lcuda; the pip toolkit has no stub, so point lib64/stubs/libcuda.so at the host driver.
drv = next(iter(sorted(glob.glob('/usr/local/nvidia/lib64/libcuda.so*')) + sorted(glob.glob('/usr/lib/x86_64-linux-gnu/libcuda.so*'))), None)
os.makedirs(f'{shim}/lib64/stubs', exist_ok=True)
for dst in (f'{shim}/lib64/stubs/libcuda.so', f'{shim}/lib64/libcuda.so'):
    if drv and not os.path.lexists(dst):
        os.symlink(drv, dst)
log('libcuda for JIT linking:', drv)
BASE_ENV = dict(os.environ)
BASE_ENV.update(CUDA_HOME=shim, CUDA_PATH=shim, PATH=f'{V}/bin:{shim}/bin:' + os.environ['PATH'],
                LD_LIBRARY_PATH=os.environ.get('LD_LIBRARY_PATH', '') + f':{shim}/lib64',
                VLLM_USE_V2_MODEL_RUNNER='1', FLASHINFER_CUDA_ARCH_LIST='8.9', TORCH_CUDA_ARCH_LIST='8.9',
                HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', DSV41_SYNTH_INIT='1',
                # FlashInfer's sampling module cannot JIT here: the pip nvcc (13.4) fails CCCL's check against the
                # pip CUDA 13.0 headers, and the box-built .so needs GLIBCXX_3.4.32 (newer than this image's libstdc++).
                # The benchmark decodes greedily (argmax), so vLLM's PyTorch top-k/top-p path is never on the hot path.
                VLLM_USE_FLASHINFER_SAMPLER='0')
for k in ('VLLM_MARLIN_INPUT_DTYPE', 'TRITON_PTXAS_BLACKWELL_PATH', 'NCCL_P2P_LEVEL'):
    BASE_ENV.pop(k, None)
mem_gb = int(open('/proc/meminfo').read().split()[1]) / 2**20
# 4M rows x 264 B = 0.98 GiB pinned per rank and layer (7.9 GiB total), as on the box; halve it on a small host.
BASE_ENV['DSV41_ENGRAM_SYNTH_ROWS'] = '4000000' if mem_gb > 60 else '2000000'
rc, out = sh(f'nvcc --version | tail -2; {PY} -c "import torch, vllm, flashinfer; print(torch.__version__, torch.version.cuda, '
             f'torch.cuda.device_count(), vllm.__version__, flashinfer.__version__)"', timeout=900, env=BASE_ENV)
log('toolchain/imports:', out[-800:], '| host RAM GiB', round(mem_gb, 1), '| engram rows', BASE_ENV['DSV41_ENGRAM_SYNTH_ROWS'])
sh(f'uv pip freeze --python {PY} > {OUT}/pip_freeze.txt 2>&1 || {PY} -m pip freeze > {OUT}/pip_freeze.txt', env=BASE_ENV)

In [ ]:
MODEL = '/tmp/dsv41/model-e64'
def serve_cmd(extra, model=None):
    return [f'{V}/bin/vllm', 'serve', model or MODEL, '--served-model-name', 'dsv41-e64', '--host', '127.0.0.1', '--port', '8100',
            '--trust-remote-code', '--tensor-parallel-size', '4', '--distributed-executor-backend', 'mp',
            '--kv-cache-dtype', 'fp8', '--block-size', '128', '--max-model-len', '65536', '--max-num-seqs', '128',
            '--max-num-batched-tokens', '2048', '--gpu-memory-utilization', '0.88', '--no-enable-prefix-caching',
            '--engram-config', '{"cpu_offload":true}', '--load-format', 'dummy', '--tokenizer-mode', 'deepseek_v41',
            '--language-model-only', '--compilation-config', '{"cudagraph_mode":"FULL"}'] + extra

class GpuMon(threading.Thread):
    def __init__(self, path):
        super().__init__(daemon=True); self.path, self.stop = path, threading.Event()
    def run(self):
        q = 'timestamp,index,memory.used,utilization.gpu,temperature.gpu,clocks.sm,power.draw'
        with open(self.path, 'a') as fh:
            while not self.stop.is_set():
                fh.write(sh(f'nvidia-smi --query-gpu={q} --format=csv,noheader', timeout=30)[1]); fh.flush()
                self.stop.wait(2)

def ready():
    try:
        urllib.request.urlopen('http://127.0.0.1:8100/v1/models', timeout=3).read(); return True
    except Exception:
        return False

LIVE = {}  # process groups the watchdog kills at HARD_STOP_S

def watchdog():
    while time.time() - T0 < HARD_STOP_S:
        time.sleep(10)
    log('WATCHDOG: hard stop, killing', list(LIVE))
    for p in list(LIVE.values()):
        try:
            os.killpg(p.pid, signal.SIGKILL)
        except Exception:
            pass
threading.Thread(target=watchdog, daemon=True).start()

def run_bounded(cmd, timeout, env=None):
    """Shell command in its own process group; on timeout the whole group dies (subprocess.run would leave a
    pipeline's children holding the pipe)."""
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env,
                         start_new_session=True)
    LIVE[cmd[:60]] = p
    try:
        out, _ = p.communicate(timeout=timeout)
    except subprocess.TimeoutExpired:
        os.killpg(p.pid, signal.SIGKILL); out, _ = p.communicate()
        out = (out or '') + f'\nTIMEOUT after {timeout}s'
    finally:
        LIVE.pop(cmd[:60], None)
    return p.returncode, out or ''

def start(name, extra=(), env_extra=None, timeout=2700, model=None):
    d = f'{OUT}/{name}'; os.makedirs(d, exist_ok=True)
    env = dict(BASE_ENV, DSV41_SYNTH_MANIFEST=f'{d}/synth', **(env_extra or {}))
    cmd = serve_cmd(list(extra), model)
    open(f'{d}/cmd.txt', 'w').write(' '.join(cmd) + '\n' + json.dumps(env_extra or {}) + '\n')
    proc = subprocess.Popen(cmd, env=env, stdout=open(f'{d}/server.log', 'w'), stderr=subprocess.STDOUT,
                            stdin=subprocess.DEVNULL, start_new_session=True, cwd=d)
    LIVE['server'] = proc
    mon = GpuMon(f'{d}/gpu.csv'); mon.start()
    t = time.time()
    while not ready():
        time.sleep(10)
        if proc.poll() is not None or time.time() - t > timeout:
            log(name, 'SERVER FAILED', proc.poll(), sh(f'grep -E "Error|error" {d}/server.log | tail -5')[1][-1500:])
            stop(proc, mon); return None, None
    kv = sh(f"grep -o 'GPU KV cache size: [0-9,]* tokens' {d}/server.log | tail -1")[1].strip()
    log(name, f'up after {time.time() - t:.0f}s', kv)
    return proc, mon

def stop(proc, mon):
    # Our own server only: its process group (start_new_session=True), never by name.
    if proc is not None and proc.poll() is None:
        os.killpg(proc.pid, signal.SIGTERM)
        try:
            proc.wait(90)
        except subprocess.TimeoutExpired:
            os.killpg(proc.pid, signal.SIGKILL); proc.wait(30)
    LIVE.pop('server', None)
    if mon is not None:
        mon.stop.set()
    for _ in range(60):
        if sh('nvidia-smi --query-compute-apps=pid --format=csv,noheader', timeout=30)[1].strip() == '':
            break
        time.sleep(2)

def grid(name, points, window=30):
    rows = []
    for ctx, c in points:
        if time_left() < 0:
            log(name, 'out of time, skipping', ctx, c); break
        rc, out = sh(f'{PY} {KIT}/decode_window.py --ctx {ctx} --conc {c} --window {window} --seed {c} '
                     f'--json {OUT}/{name}/dw_ctx{ctx}_c{c}.json 2>&1 | tail -1', timeout=3600, env=BASE_ENV)
        try:
            r = json.loads(out.strip().splitlines()[-1]); r.pop('first_token_s', None); rows.append(r)
            log(name, f'ctx {ctx} C {c}: decode {r["decode_tok_s"]} tok/s, ITL {r["itl_median_ms"]} ms, '
                      f'prefill {r["prefill_tok_s"]} tok/s')
        except Exception:
            log(name, 'grid point failed', ctx, c, out[-800:])
    json.dump(rows, open(f'{OUT}/{name}/grid.json', 'w'), indent=1)
    return rows

def smoke(name):
    body = json.dumps({'model': 'dsv41-e64', 'prompt': 'The quick brown fox jumps over the lazy dog. Count to ten:',
                       'max_tokens': 32, 'temperature': 0, 'ignore_eos': True, 'logprobs': 1}).encode()
    req = urllib.request.Request('http://127.0.0.1:8100/v1/completions', body, {'Content-Type': 'application/json'})
    d = json.loads(urllib.request.urlopen(req, timeout=600).read())
    lp = d['choices'][0]['logprobs']['token_logprobs']
    import math
    log(name, 'smoke: tokens', d['usage']['completion_tokens'], 'logprobs finite', all(math.isfinite(x) for x in lp))

In [ ]:
# Runs 6-8: realistic tool mode. E030 attempt shapes (sub80s40 + header): slots start mid-attempt, run attempts back
# to back with a 1 s tool delay; the window fills what is left before the hard stop. PR #154, prefix caching on.
# Run 9: E64 x 96, then E64 x 80 (run 8's E64 x 64 peaked at 73% KV, so 64 under-fills it vs run 7's E48 x 128).
# Each point gets a fresh server (empty prefix cache) and run 8's replay settings; (model, C, replay wall cap or None
# for whatever is left). Run 8: E64 x 64; runs 6-7: E48 x 128.
TOOLS_POINTS = [('e64', 96, 5200), ('e64', 80, None)]
POINT_MIN_S = 4000  # server start + setup + drain + warm-up + a 20-min window at C 80
class KvMon(threading.Thread):
    KEYS = ('vllm:kv_cache_usage_perc', 'vllm:num_requests_running', 'vllm:num_requests_waiting',
            'vllm:num_preemptions_total', 'vllm:prefix_cache_queries_total', 'vllm:prefix_cache_hits_total',
            'vllm:generation_tokens_total')
    def __init__(self, path):
        super().__init__(daemon=True); self.path, self.stop = path, threading.Event()
    def run(self):
        with open(self.path, 'a') as fh:
            while not self.stop.is_set():
                try:
                    txt = urllib.request.urlopen('http://127.0.0.1:8100/metrics', timeout=10).read().decode()
                    vals = {}
                    for line in txt.splitlines():
                        for k in self.KEYS:
                            if line.startswith(k + '{') or line.startswith(k + ' '):
                                vals[k[5:]] = vals.get(k[5:], 0.0) + float(line.split()[-1])
                    fh.write(json.dumps({'t': round(time.time() - T0), **vals}) + '\n'); fh.flush()
                except Exception as exc:
                    fh.write(json.dumps({'t': round(time.time() - T0), 'error': repr(exc)[:200]}) + '\n')
                self.stop.wait(30)

SUMMARY = {}
for TOOLS_MODEL, TOOLS_C, cap in TOOLS_POINTS:
    name = f'tools_{TOOLS_MODEL}_c{TOOLS_C}'; d = f'{OUT}/{name}'
    if time_left() < POINT_MIN_S:
        log(name, f'skipped: {time_left():.0f}s left'); continue
    SUMMARY[name] = []
    # --prefix-cache-retention-interval 128: keep every sliding-window block, so blocks filled by generated tokens are
    # reusable (the default 0 keeps SWA KV only at the prompt end, and a re-sent transcript hits only the old prompt; P7).
    proc, mon = start(name, extra=['--enable-prefix-caching', '--max-model-len', '163840', '--prefix-cache-retention-interval', '128'],
                      env_extra={'DSV41_INDEXER_Q_PREDECODE': '1'}, model=f'/tmp/dsv41/model-{TOOLS_MODEL}', timeout=1500)
    kvm = None
    try:
        if proc is not None:
            smoke(name)
            kvm = KvMon(f'{d}/kvmon.log'); kvm.start()
            budget = time_left() - 240
            if cap:
                budget = min(budget, cap)
            log(name, f'tool replay C {TOOLS_C}, wall budget {budget:.0f}s')
            rc, out = run_bounded(f'{PY} {KIT}/tool_replay.py --shapes {KIT}/tools_e030_hdrmin.json --conc {TOOLS_C} --warm-s 300 '
                                  f'--window-s 2400 --tool-delay 1 --seed 1 --max-wall-s {budget:.0f} '
                                  f'--json {d}/r2_tools_c{TOOLS_C}.json 2>&1 | tail -1', timeout=budget + 180, env=BASE_ENV)
            try:
                r = json.loads(out.strip().splitlines()[-1]); SUMMARY[name].append(r)
                log(name, 'tool replay:', json.dumps(r))
            except Exception:
                log(name, 'tool replay failed', out[-1500:])
            json.dump(SUMMARY, open(f'{OUT}/summary.json', 'w'), indent=1)
    except Exception as exc:
        log(name, 'ERROR', repr(exc))
    finally:
        if kvm is not None:
            kvm.stop.set()
        stop(proc, mon)

In [ ]:
for d in glob.glob(f'{OUT}/*/server.log'):
    sh(f'gzip -f {d}')
log('E042 KAGGLE DONE', json.dumps(SUMMARY)[:3000])